**Project:** Data Mining II (2025/26)

**Group Number:** 12

**Members:**
- Beatriz Boura - 20250272
- Dinis Gaspar - 20221869
- Margarida Cruz - 20221929

**Project Overview**

In recent years, nonprofit organizations have faced a growing challenge: while charitable causes have multiplied, public tolerance for repeated, generic solicitations has significantly decreased, often leading to donor fatigue and long-term disengagement. To address this issue, the Civic Support Alliance (CSA)—a federation representing multiple humanitarian and social aid programs—seeks to modernize its fundraising strategy. Rather than launching blanket campaigns across their entire database, the organization aims to transition to a highly targeted approach. The goal is to maximize operational efficiency and maintain donor respect by contacting fewer, but more receptive, individuals. 

As data scientists, our team has been tasked with building a predictive machine learning system using historical demographic, interaction, and donation data accumulated from past campaigns. The primary objective is to accurately answer a fundamental question: Will this person donate if contacted? 

**Notebook Introduction**

In this notebook, we will develop Naive Bayes models, we will find parameter regions using our hold-out method splits as explained in teh Modeling Tools notebook to test and we will then perform a parameter search to try and maximize performances.

**Benchmarks**

As the goal of this project is to help the CSA create and improve a targeted approach to donors with the goal of maximizing donations, we will use 2 baseline benchmarks as the minimum any model must achieve to be a good model:
+ Random prediction, which in a dataset with the imbalance present in the CSA's data yield an **F1-Score of ~0.34**
+ Predicting all 1, which is essentially, telling the CSA to keep sending out the camapaign to everyone. This is obviously not the desire of the CSA and as such the goal is to create models that can avoid this. Predicting all 1 in a dataset with this imbalance yields an **F1-Score of 0.4**

# 1. <a id='toc1_'></a>[Imports](#toc0_)

In this section we're importing everything we need, libraries and tools.

In [11]:
from sklearnex import patch_sklearn
patch_sklearn()
from utils_modeling import (OutlierClipper, CategoricalFeatureSelector, NumericalFeatureSelector, FeatureEngineer, DataCleaner, run_parameter_search)
import os
if os.getcwd() != 'c:\\Users\\dinis\\OneDrive\\Ambiente de Trabalho\\Faculdade - MGI-BI\\1º ano\\2º Semestre\\Data Mining II\\Project\\DM2_Project':
    %cd ..
import numpy as np
import pandas as pd
from sklearn.model_selection import TunedThresholdClassifierCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, MinMaxScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings("ignore")
SEED=23
pd.options.display.max_columns = None

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [12]:
train = pd.read_csv('Files/donors_train.csv')
with open('Files/Pickle Files/model_testing_skf.pkl', 'rb') as file:
    model_testing_skf = pickle.load(file)
with open('Files/Pickle Files/data_cleaner.pkl', 'rb') as file:
    data_cleaner = pickle.load(file)
with open('Files/Pickle Files/X_train_preprocessed.pkl', 'rb') as file:
    X_train_preprocessed = pickle.load(file)
with open('Files/Pickle Files/X_val_preprocessed.pkl', 'rb') as file:
    X_val_preprocessed = pickle.load(file)
with open('Files/Pickle Files/y_train.pkl', 'rb') as file:
    y_train = pickle.load(file)
with open('Files/Pickle Files/y_val.pkl', 'rb') as file:
    y_val = pickle.load(file)

In [13]:
X = train.drop('TARGET_B', axis=1)
y = train['TARGET_B']

# 2. <a id='toc2_'></a>[Defining the Pipeline](#toc0_)

We're first going to start by defining the pipeline we're gonna use as a base. This is the pipeline introduced in the Modeling Tools notebook, now with a Multinomial Naive Bayes model, we use GaussianNB (which expects normally distributes features, but works even if they're not) since Multinomial NB can't handle negative values. We also change the scaler step to the PowerTransformer since GaussianNB expects normally distributed variables.

In [ ]:
# Categorical Feature Sub-Pipeline
# This is the part that handles the categorical columns, performing
# mode imputation, feature selection using our custom CategoricalFeatureSelector
# and finally one-hot encoding the features.
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('feature_selection',  CategoricalFeatureSelector()),
    # 3. Your specialized encoding (OneHot/Target) now receives imputed integers
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

# Numerical Feature Sub-Pipeline
# Here we take care of our numerical features, starting with outlier clipping and feature
# creation using our custom transformer, then scaling data, so that it can then 
# be imputed and numerical feature selection can be performed
num_pipe = Pipeline([
    ('clipper', OutlierClipper()),
    ('feature_engineer', FeatureEngineer()),
    ('scaler', PowerTransformer()),
    ('imputer', KNNImputer()),
    ('feature_selection', NumericalFeatureSelector(random_state=SEED))
])

# Here we use a ColumnTransformer with column selectors to perform the split
# between numerical and categorical data, so that each subset can be directed
# to the appropriate sub-pipeline
preprocessor = ColumnTransformer([
    ('cat_section', cat_pipe, make_column_selector(dtype_exclude=[np.number])),
    ('num_section', num_pipe, make_column_selector(dtype_include=[np.number])),
],
verbose_feature_names_out=False)
#  

 
# Final Pipeline
nb_pipe = Pipeline([
    ('cleaner', data_cleaner),
    ('preprocessing', preprocessor),
    ('model', TunedThresholdClassifierCV(GaussianNB(),
                                         n_jobs=-1,
                                         scoring='f1',
                                         random_state=SEED,
                                         cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
                                         )) 
])

# We use the set_output to pandas so that intermediate transformers can use column
# names, since some of the default scikit-learn transformers by default return 
# Numpy arrays, which obviously don't have column names.
nb_pipe.set_output(transform="pandas")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,categorical_cols_values,"{'DONOR_GENDER': ['M', 'F', ...], 'INCOME_GROUP': array([1, 2, 3, 4, 5, 6, 7]), 'PEP_STAR': [0, 1], 'RECENCY_STATUS_96NK': ['S', 'A', ...], ...}"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_section', ...), ('num_section', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that

Since the GaussianNB model only has 1 parameter that is commonly tuned, which is the smoothing parameter, we're going to move straight to the parameter search.

# Parameter search

We're going to test various values for the smoothing parameter as well as some diferent preprocessing combinations. The full breakdown is below:
+ In Preprocessing
    + Scaler: We are going to test exclusively the PowerTransformer. This transformer applies transformations to correct highly skewed data into distributions that are as close to a Gaussian Normal distribution as possible. Additionally, it automatically standardizes the transformed data to have a mean of 0 and a variance of 1, effectively satisfying the core assumption of our classifier without needing an extra scaling step. Other scalers are not relevant since GaussianNB expects normally distributed variables and MinMax or Robust scaler don't perform any transformations to skewed data.
    + Imputer: We will evaluate two robust imputation techniques: the distance-based KNNImputer configured with 50 neighbors, and the IterativeImputer. The IterativeImputer uses a round-robin approach to model each missing feature as a function of all other features via its base BayesianRidge regression estimator, allowing us to find out whether spatial proximity or multivariate regression handles our missing values best.
    + Clipping Strategies: Three outlier-handling approaches will be tested: no clipping at all (None), clipping based on the standard IQR method for moderate outliers (1.5 times the IQR beyond the quartiles), and clipping for extreme outliers (3 times the IQR beyond the quartiles). This allows us to see how variance restriction interacts with the smoothing parameter of the model.

+ In the Model
    + Variance Smoothing (var_smoothing): Since GaussianNB relies heavily on stable feature variances to calculate its Gaussian probability density functions, we will thoroughly test the var_smoothing parameter. This parameter adds a user-defined portion of the dataset's maximum feature variance to all feature variances to prevent calculation instability. We will perform a deep grid search using 25 logarithmically spaced values spanning from 1.0 (highly aggressive smoothing that flattens the distribution curves) all the way down to $10^{-9}$ (the standard scikit-learn baseline default).

In [23]:
param_grid = {
    'preprocessing__num_section__scaler' : [PowerTransformer()], 
    'preprocessing__num_section__imputer' : [KNNImputer(n_neighbors=50), IterativeImputer(random_state=SEED, max_iter=10, initial_strategy='median')], 
    'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
    'model__estimator' : [GaussianNB()],
    'model__estimator__var_smoothing' : np.logspace(0, -9, num=25) 
}

In [24]:
run_parameter_search(grid=param_grid,
                     cv=model_testing_skf, 
                     X=X, y=y,
                     model=nb_pipe,
                     metrics=['f1', 'precision', 'recall'],
                     results_file_dir='Files/Pickle Files/Results/NB_GridSearch_Results.pkl',
                     model_file_dir='Files/Pickle Files/Models/NB_GridSearch_Best_Model.pkl',
                     refit=True,
                     n_jobs=-1)

Tuning Hyperparameters:   0%|          | 0/150 [00:00<?, ?it/s]

(                                        params_config  \
 6   {'model__estimator': GaussianNB(var_smoothing=...   
 35  {'model__estimator': GaussianNB(var_smoothing=...   
 20  {'model__estimator': GaussianNB(var_smoothing=...   
 22  {'model__estimator': GaussianNB(var_smoothing=...   
 19  {'model__estimator': GaussianNB(var_smoothing=...   
 ..                                                ...   
 73  {'model__estimator': GaussianNB(var_smoothing=...   
 59  {'model__estimator': GaussianNB(var_smoothing=...   
 39  {'model__estimator': GaussianNB(var_smoothing=...   
 42  {'model__estimator': GaussianNB(var_smoothing=...   
 45  {'model__estimator': GaussianNB(var_smoothing=...   
 
                                      model__estimator  \
 6   GaussianNB(var_smoothing=np.float64(0.42169650...   
 35  GaussianNB(var_smoothing=np.float64(0.42169650...   
 20  GaussianNB(var_smoothing=np.float64(0.42169650...   
 22  GaussianNB(var_smoothing=np.float64(0.42169650...   
 19  Gaussia

In [27]:
result_df = pd.read_pickle('Files/Pickle Files/Results/NB_GridSearch_Results.pkl')
result_df.head()

,params_config,model__estimator,model__estimator__var_smoothing,preprocessing__num_section__clipper,preprocessing__num_section__imputer,preprocessing__num_section__scaler,mean_fit_time,mean_val_f1,std_val_f1,mean_train_f1,std_train_f1,mean_val_precision,std_val_precision,mean_train_precision,std_train_precision,mean_val_recall,std_val_recall,mean_train_recall,std_train_recall,status
6,{'model__estimator': GaussianNB(var_smoothing=...,GaussianNB(var_smoothing=np.float64(1e-09)),0.421697,None,KNNImputer(n_neighbors=50),PowerTransformer(),65.671318,0.419758,0.007662,0.419638,0.001944,0.293715,0.004062,0.293469,0.002378,0.735693,0.025774,0.736283,0.009042,Success
35,{'model__estimator': GaussianNB(var_smoothing=...,GaussianNB(var_smoothing=np.float64(1e-09)),0.013335,OutlierClipper(iqr_multiplier=3),"IterativeImputer(initial_strategy='median', ra...",PowerTransformer(),134.333716,0.419105,0.009187,0.419018,0.002470,0.290398,0.006705,0.290223,0.003372,0.752802,0.016119,0.753835,0.015789,Success
20,{'model__estimator': GaussianNB(var_smoothing=...,GaussianNB(var_smoothing=np.float64(1e-09)),0.074989,OutlierClipper(),KNNImputer(n_neighbors=50),PowerTransformer(),58.823243,0.419028,0.006269,0.418835,0.001719,0.290574,0.003575,0.290257,0.003568,0.751622,0.025929,0.752729,0.020379,Success
22,{'model__estimator': GaussianNB(var_smoothing=...,GaussianNB(var_smoothing=np.float64(1e-09)),0.074989,OutlierClipper(iqr_multiplier=3),KNNImputer(n_neighbors=50),PowerTransformer(),71.058513,0.418983,0.004622,0.419887,0.002311,0.292372,0.002824,0.292986,0.006090,0.741003,0.040592,0.742625,0.029847,Success
19,{'model__estimator': GaussianNB(var_smoothing=...,GaussianNB(var_smoothing=np.float64(1e-09)),0.074989,None,"IterativeImputer(initial_strategy='median', ra...",PowerTransformer(),167.563547,0.418822,0.006722,0.418936,0.001751,0.289284,0.008125,0.289611,0.007296,0.761947,0.044665,0.760546,0.042352,Success


In [28]:
result_df.iloc[0]['params_config']

{'model__estimator': GaussianNB(var_smoothing=np.float64(1e-09)),
 'model__estimator__var_smoothing': np.float64(0.4216965034285822),
 'preprocessing__num_section__clipper': None,
 'preprocessing__num_section__imputer': KNNImputer(n_neighbors=50),
 'preprocessing__num_section__scaler': PowerTransformer()}

In [29]:
with open('Files/Pickle Files/Models/NB_GridSearch_Best_Model.pkl', 'rb') as file:
    nb_bestmodel = pickle.load(file)

In [30]:
nb_bestmodel['model'].best_threshold_

np.float64(0.11176396329168903)

The best parameter combination achieved a mean validation F1-score of 0.419758 and a mean training F1-score of 0.419638 indicating no over or under fitting. It used the Power Transformation (as expected since it's the only option), no Outlier clipping and the KNNImputer. The model used a amoothing value of 0.4216965034285822.

Additionally, it has an optimized decision threshold of ~0.11 which means it allows for a lot more predicted positives, in other words predicts many more donations than if it were using the default threshold. This means that the recall is higher using this threshold which means that the model is finding a higher rate of the true donors. This comes with the side effect of also sending the campaign to more people who won't donate (False Postives) lowering precision. But as the goal is to maximize donations while optimizing the campaign distribution process, this seems like a decent compromise.

# Test Set Prediction

The last thing to do is to predict for the test set. For that we'll read the test set and the pickle file of the best model and then generate the prediction and export it to a CSV file as instructed.

In [31]:
test = pd.read_csv('Files/donors_test.csv')


In [32]:
pred_test = pd.DataFrame(nb_bestmodel.predict(test), index=test['CONTROL_NUMBER'], columns=['TARGET_B'])
pred_test

,TARGET_B
CONTROL_NUMBER,
122653,0
184239,1
5172,1
135377,1
62119,1
...,...
54438,1
122194,1
106603,1


In [33]:
pred_test.to_csv('Files/Submissions/DM2DT_Group12_Version20.csv')

The best model from the Naive Bayes parameter search achieves an F1-Score of 0.42893 as the public score on Kaggle.